![Module 5: Ambient / Autonomous Mode](../images/module-5-ambient.svg)

# Module 5: Ambient / Autonomous Mode

> The agent keeps working when you step away. Idle detection triggers background iterations on the last topic. Findings auto-inject into your next query.

📖 Full explainer: see the companion page in `content/` of this repo.  
💻 Standalone script: `code/step-0?/agent.py`  
🔗 Workshop: https://github.com/strands-agents/samples/tree/main/python/01-learn/18-self-improving-agents

---


# AIM308 — Module 4: Autonomous (Ambient Mode)

In [ ]:
%pip install -q strands-agents strands-agents-tools bedrock-agentcore

In [ ]:
import os

MODEL_ID = "global.anthropic.claude-opus-4-8"  # Claude Sonnet 4.5
os.environ["AWS_REGION"] = "us-east-1"


## The ambient loop

In [ ]:
import threading, timeclass AmbientMode:    def __init__(self, agent_factory, idle_seconds=15, max_iterations=2):        self.agent_factory = agent_factory        self.idle_seconds = idle_seconds        self.max_iterations = max_iterations        self.last_interaction = time.time()        self.last_query = None        self.pending = None        self.iterations = 0        self.running = False    def record(self, q):        self.last_interaction = time.time()        self.last_query = q        self.iterations = 0    def consume(self):        r, self.pending = self.pending, None        return r    def _loop(self):        while self.running:            idle = time.time() - self.last_interaction            if (self.last_query and idle > self.idle_seconds                     and self.iterations < self.max_iterations):                prompt = f"Continue exploring: '{self.last_query}'. Find edges, extensions. Be concise."                print(f"🌙 [ambient iter {self.iterations+1}/{self.max_iterations}] thinking...")                r = self.agent_factory()(prompt)                self.pending = (self.pending or "") + "\n\n" + str(r)                self.iterations += 1            time.sleep(3)    def start(self):        self.running = True        threading.Thread(target=self._loop, daemon=True).start()

## Use itAsk a question → wait 15s → consume pending results.

In [ ]:
from strands import Agentfrom strands_tools import shelldef factory():    return Agent(model=MODEL_ID, tools=[shell])amb = AmbientMode(factory, idle_seconds=15, max_iterations=2)amb.start()# ask somethingfactory()("What are the top 3 techniques to speed up LLM inference?")amb.record("What are the top 3 techniques to speed up LLM inference?")

In [ ]:
# wait ~40 seconds, then run this cellimport time; time.sleep(40)pending = amb.consume()print(f"Ambient produced {len(pending or '')} chars\n")print((pending or "")[:800])

Next: **[05_deploy.ipynb](05_deploy.ipynb)**